<a href="https://colab.research.google.com/github/sarsa1/ml-titanic-baseline/blob/main/Clasificador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Primer modelo de ML, clasificador, con limpieza de datos incluida.

**Primero, cargamos los datos con pandas**

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer

try:
     df = pd.read_csv("titanic.csv")
except:
  df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")



**Segundo, ¿Cúantos valores nulos tiene cada atributo? Para saber que limpiar**

In [40]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


*Como bien podemos observar, No sabemos donde dos personas embarcamos, y la edad tampoco*
*¿Qué voy a hacer yo?*


1.   *Primero: las personas cuya entidad edad sea nula; le asignaremos la media de todas las edades*
2.   *Segundo: Solo hay dos personas que no se sabe su valor de embarque, se asignará al más frequente también*




In [41]:
#Asignación de valor medio de la edad a las personas
Imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
df[['Age']] = Imputer.fit_transform(df[['Age']])


In [42]:
Imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
df[['Embarked']] = Imputer.fit_transform(df[['Embarked']])

**Ya no hay valores nulos, pero deberemos asignar valores numericos a esos atributos que no los tengan**

In [43]:
df[['Sex']] = OrdinalEncoder().fit_transform(df[['Sex']]) # 1 será hombres y 0 mujeres
print(df[['Sex']])
df[['Embarked']] = OrdinalEncoder().fit_transform(df[['Embarked']]) # 2: S ,1: Q, 0: C

     Sex
0    1.0
1    0.0
2    0.0
3    0.0
4    1.0
..   ...
886  1.0
887  0.0
888  0.0
889  1.0
890  1.0

[891 rows x 1 columns]


**Haré un drop a la columna Name, Ticket y a la cabina. No son útiles**

In [44]:
df=df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

**Normalizamos la edad y el Fare**

In [45]:
df[['Age']] = MinMaxScaler().fit_transform(df[['Age']])
df[['Fare']] = StandardScaler().fit_transform(df[['Fare']])
print([df[['Age', 'Fare']]])

[          Age      Fare
0    0.271174 -0.502445
1    0.472229  0.786845
2    0.321438 -0.488854
3    0.434531  0.420730
4    0.434531 -0.486337
..        ...       ...
886  0.334004 -0.386671
887  0.233476 -0.044381
888  0.367921 -0.176263
889  0.321438 -0.044381
890  0.396833 -0.492378

[891 rows x 2 columns]]


In [46]:
#Como va nuestra tabla ahora mismo...
print(df)

     Survived  Pclass  Sex       Age  SibSp  Parch      Fare  Embarked
0           0       3  1.0  0.271174      1      0 -0.502445       2.0
1           1       1  0.0  0.472229      1      0  0.786845       0.0
2           1       3  0.0  0.321438      0      0 -0.488854       2.0
3           1       1  0.0  0.434531      1      0  0.420730       2.0
4           0       3  1.0  0.434531      0      0 -0.486337       2.0
..        ...     ...  ...       ...    ...    ...       ...       ...
886         0       2  1.0  0.334004      0      0 -0.386671       2.0
887         1       1  0.0  0.233476      0      0 -0.044381       2.0
888         0       3  0.0  0.367921      1      2 -0.176263       2.0
889         1       1  1.0  0.321438      0      0 -0.044381       0.0
890         0       3  1.0  0.396833      0      0 -0.492378       1.0

[891 rows x 8 columns]





1.   *data_x: Los valores sin el resultado a predecir*
2.   *data_y: solo los valores que se han predecido*



In [48]:
data_x = df.drop(columns=['Survived'])
data_y = df['Survived']

print('Feature DataFrame')
display(data_x.shape)
print('Target DataFrame')
display(data_y.shape)

Feature DataFrame


(891, 7)

Target DataFrame


(891,)

**Ahora dividiremos el espacio en 80% entrenamiento y 20% para testear el modelo; de ese 80% de entreno querré un 20% de validación**

In [51]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2
RANDOM_STATE = 0
#20% reservado para examen final
X_train, X_test, Y_train, Y_test = train_test_split(data_x, data_y,
                                                    test_size=TEST_SIZE,
                                                    random_state=RANDOM_STATE)
#80% reservado para entrenar, con un 20% para validacion
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train,
                                                  test_size=TEST_SIZE,
                                                  random_state=RANDOM_STATE)

print('DATOS SIN LO QUE QUIERO PREDECIR, LOS FEATURES')
print('Training size: ' + str(X_train.shape))
print('Validation size: ' + str(X_val.shape))
print('Test size: ' + str(X_test.shape))

print('DATOS CON LO QUE QUIERO PREDECIR')
print('Training size: ' + str(Y_train.shape))
print('Validation size: ' + str(Y_val.shape))
print('Test size: ' + str(Y_test.shape))

DATOS SIN LO QUE QUIERO PREDECIR, LOS FEATURES
Training size: (569, 7)
Validation size: (143, 7)
Test size: (179, 7)
DATOS CON LO QUE QUIERO PREDECIR
Training size: (569,)
Validation size: (143,)
Test size: (179,)


**Al iniciar la semilla de random state lo que he hecho es escoger un numero aleatorio de supervivientes y no supervivientes. Ahora, veré que proporciones se han cogido**

In [53]:
def get_proportions(data):

    counts = pd.DataFrame(np.unique(data, return_counts=True), index = ['Survived', 'Counts']).T
    counts['Percent'] = (counts['Counts']/counts['Counts'].sum()).round(2)*100
    counts = counts.set_index('Survived')
    return counts


print('Training DataFrame')
display(get_proportions(Y_train))
print('Test DataFrame')
display(get_proportions(Y_test))

Training DataFrame


,Counts,Percent
Survived,,
0,356,63.0
1,213,37.0


Test DataFrame


,Counts,Percent
Survived,,
0,110,61.0
1,69,39.0


**Es necesario escalar para clasificadores**

In [60]:
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)

**Entrenamos el modelo**

In [61]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=RANDOM_STATE)
lr.fit(X_train_scale, Y_train)

LogisticRegression(random_state=0)

**Hagamos que el modelo prediga si la gente sobrevivirá o no**

In [62]:
X_val_scale = scaler.transform(X_val) # fit transform no, para no contaminar
Y_predict = lr.predict(X_val_scale)
print(Y_predict)

[0 1 0 0 1 1 1 1 0 0 0 0 0 0 1 1 1 1 1 1 0 1 0 0 0 0 1 0 1 0 0 0 1 1 0 0 0
 0 1 0 0 1 1 0 1 1 0 0 0 0 1 1 1 1 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0
 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 0 1 0 0 0 0
 0 1 1 0 0 1 1 1 0 0 0 1 0 1 0 1 1 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0]


**¿Cual habrá sido la precisión de nuestro modelo?**

In [63]:
from sklearn.metrics import accuracy_score

precision = accuracy_score(Y_val, Y_predict)

print(f"Precisión modelo: {precision * 100:.2f}%")

Precisión modelo: 76.22%
